In [2]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from typing import List, Tuple

# ---------- Read SExtractor Catalog ----------
def _get_column_names(read_line_object: List[str]) -> List[str]:
    return [line.split()[2] for line in read_line_object if line.startswith('#')]

def _get_rows(read_line_object: List[str]) -> List[List[float]]:
    return [list(map(float, line.split())) for line in read_line_object if not line.startswith('#')]

def split_names_and_data(read_line_object: List[str]) -> Tuple[List[str], List[List[float]]]:
    header = _get_column_names(read_line_object)
    data = _get_rows(read_line_object)
    return header, data

def read_cat(sextractor_catalog: str) -> pd.DataFrame:
    with open(sextractor_catalog, encoding='utf8') as file:
        lines = file.readlines()
    column_names, data = split_names_and_data(lines)
    df = pd.DataFrame(data, columns=column_names)
    print(f"Loaded catalog with columns: {df.columns.tolist()}")
    return df

# ---------- Find Closest DECam Object ----------
def find_closest_objects(
    decam_df: pd.DataFrame,
    target_coords: List[Tuple[str, str]],
    max_sep_arcsec: float = 1.0
) -> pd.DataFrame:
    # Ensure required columns exist
    required_cols = ['ALPHA_J2000', 'DELTA_J2000', 'MAG_AUTO', 'MAGERR_AUTO']
    for col in required_cols:
        if col not in decam_df.columns:
            raise KeyError(f"Column '{col}' not found in SExtractor catalog!")

    # SkyCoord for all catalog detections
    decam_coords = SkyCoord(
        ra=decam_df['ALPHA_J2000'].values * u.deg,
        dec=decam_df['DELTA_J2000'].values * u.deg
    )

    # Generate target names
    target_names = ["QSO"] + [f"LAE-{i}" for i in range(1, len(target_coords))]

    results = []
    for i, (ra_str, dec_str) in enumerate(target_coords):
        target = SkyCoord(ra=ra_str, dec=dec_str, unit=(u.hourangle, u.deg))
        idx, sep2d, _ = target.match_to_catalog_sky(decam_coords)

        # If separation > max allowed, mark as non-detected
        if sep2d.arcsec > max_sep_arcsec:
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': np.nan,
                'Matched_Dec_deg': np.nan,
                'Separation_arcsec': np.nan,
                'MAG_AUTO': np.nan,
                'MAGERR_AUTO': np.nan,
                'Detected': False
            })
        else:
            closest_obj = decam_df.iloc[idx]
            results.append({
                'Target': target_names[i],
                'Input_RA_deg': target.ra.deg,
                'Input_Dec_deg': target.dec.deg,
                'Matched_RA_deg': closest_obj['ALPHA_J2000'],
                'Matched_Dec_deg': closest_obj['DELTA_J2000'],
                'Separation_arcsec': sep2d.arcsec,
                'MAG_APER': closest_obj['MAG_APER']+30.243,  # Adjusted MAG_APER
                'MAGERR_APER': closest_obj['MAGERR_APER'],
                'Detected': True
            })

    return pd.DataFrame(results)

# ---------- Example Usage ----------
if __name__ == '__main__':
    cat_path = '/Users/aishwarya/Documents/Lyman_alpha/CAT/n964_band_final_nbmasked.cat'  

    input_targets = [
        ("23:48:33.33", "-30:54:10.23"),  # QSO
        ("23:50:39.46", "-31:47:26.41"),  # LAE-1
        ("23:49:25.39", "-31:46:34.95"),  # LAE-2
        ("23:49:45.53", "-31:39:01.46"),  # LAE-3
        ("23:48:22.23", "-31:37:53.92"),  # LAE-4
        ("23:48:17.08", "-31:30:18.94"),  # LAE-5
        ("23:46:51.65", "-31:19:29.93"),  # LAE-6
        ("23:52:04.95", "-31:16:20.60"),  # LAE-7
        ("23:44:51.34", "-31:14:01.78"),  # LAE-8
        ("23:49:09.11", "-31:11:23.68"),  # LAE-9
        ("23:48:38.18", "-31:10:25.49"),  # LAE-10
        ("23:52:11.44", "-31:10:09.82"),  # LAE-11
        ("23:50:09.67", "-31:00:28.43"),  # LAE-12
        ("23:52:50.08", "-30:59:23.05"),  # LAE-13
        ("23:46:52.33", "-30:57:07.02"),  # LAE-14
        ("23:52:39.27", "-30:51:50.73"),  # LAE-15
        ("23:51:42.32", "-30:50:46.97"),  # LAE-16
        ("23:45:30.76", "-30:50:42.20"),  # LAE-17
        ("23:45:38.56", "-30:41:06.23"),  # LAE-18
        ("23:50:56.96", "-30:40:16.95"),  # LAE-19
        ("23:46:20.25", "-30:40:03.94"),  # LAE-20
        ("23:48:12.27", "-30:32:34.32"),  # LAE-21
        ("23:44:45.07", "-30:31:07.12"),  # LAE-22
        ("23:47:37.50", "-30:28:28.35"),  # LAE-23
        ("23:44:31.64", "-30:27:35.79"),  # LAE-24
        ("23:47:38.55", "-30:25:47.75"),  # LAE-25
        ("23:44:36.91", "-30:25:04.43"),  # LAE-26
        ("23:48:21.02", "-30:24:04.19"),  # LAE-27
        ("23:47:21.19", "-30:22:53.54"),  # LAE-28
        ("23:46:27.28", "-30:21:51.57"),  # LAE-29
        ("23:49:27.05", "-30:21:25.98"),  # LAE-30
        ("23:50:27.20", "-30:21:13.39"),  # LAE-31
        ("23:45:02.91", "-30:12:32.87"),  # LAE-32
        ("23:49:34.65", "-30:12:40.49"),  # LAE-33
        ("23:50:23.24", "-30:10:45.41"),  # LAE-34
        ("23:50:45.07", "-30:05:13.35"),  # LAE-35
        ("23:49:43.48", "-30:03:01.26"),  # LAE-36
        ("23:50:36.54", "-30:01:46.79"),  # LAE-37
        ("23:48:43.58", "-29:53:13.94"),  # LAE-38
    ]

    decam_df = read_cat(cat_path)
    closest_df = find_closest_objects(decam_df, input_targets, max_sep_arcsec=1.0)

    # Show all matches
    print("\nClosest DECam Detections:")
    print(closest_df.to_string(index=False))
    # Count True detections
    num_detected = closest_df['Detected'].sum()
    print(f"Number of true detections: {num_detected}")

    # Save to CSV for inspection
    closest_df.to_csv('n964_matched_targets.csv', index=False)
    print (len(closest_df), "matches found.")
    print("\nSaved results to n964_matched_targets.csv")


Loaded catalog with columns: ['NUMBER', 'X_IMAGE', 'Y_IMAGE', 'ALPHA_J2000', 'DELTA_J2000', 'MAG_APER', 'MAGERR_APER', 'MAG_AUTO', 'MAGERR_AUTO', 'FLAGS']

Closest DECam Detections:
Target  Input_RA_deg  Input_Dec_deg  Matched_RA_deg  Matched_Dec_deg     Separation_arcsec  MAG_APER  MAGERR_APER  Detected  MAG_AUTO  MAGERR_AUTO
   QSO    357.138875     -30.902842      357.138891       -30.902810 [0.12412978382819856]   22.1044       0.0152      True       NaN          NaN
 LAE-1    357.664417     -31.790669      357.664483       -31.790699 [0.23082356342783664]   25.3851       0.1336      True       NaN          NaN
 LAE-2    357.355792     -31.776375      357.355753       -31.776371 [0.11939074103813041]   25.7555       0.1797      True       NaN          NaN
 LAE-3    357.439708     -31.650406             NaN              NaN                   NaN       NaN          NaN     False       NaN          NaN
 LAE-4    357.092625     -31.631644      357.092637       -31.631624 [0.08145376574